# Report dataset sizes

This notebook reports file sizes for the external beach-cell dataset without reading matrix contents. It uses filesystem metadata only. It prints the total external dataset size, per-date folder sizes, and the size of files corresponding to annotated cells from `Data/Raw/Descriptor_fs.csv`.

In [ ]:
from pathlib import Path
import os
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").exists()
    or (PROJECT_ROOT / "Data").exists()
    or (PROJECT_ROOT / "pmaciag-volume-beach").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if (PROJECT_ROOT / "pmaciag-volume-beach").exists():
    PROJECT_ROOT = PROJECT_ROOT / "pmaciag-volume-beach"
if not ((PROJECT_ROOT / "Source").exists() or (PROJECT_ROOT / "Data").exists()):
    raise RuntimeError("Could not locate pmaciag-volume-beach project directory.")

CELL_DATA_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
DESCRIPTOR_PATH = PROJECT_ROOT / "Data" / "Raw" / "Descriptor_fs.csv"
FEATURE_SUFFIXES = ("r", "g", "b", "s", "d")

print(f"Project root:      {PROJECT_ROOT}")
print(f"External dataset:  {CELL_DATA_ROOT}")
print(f"Descriptor:        {DESCRIPTOR_PATH}")

In [ ]:
def format_size(num_bytes):
    """Return both decimal TB/GB and binary TiB/GiB for clarity."""
    num_bytes = int(num_bytes)
    return (
        f"{num_bytes / 1_000_000_000_000:.4f} TB "
        f"({num_bytes / 1024**4:.4f} TiB, "
        f"{num_bytes / 1_000_000_000:.2f} GB)"
    )

def iter_files(root):
    """Yield files recursively using os.scandir for lower overhead than pathlib.rglob."""
    stack = [Path(root)]
    while stack:
        current = stack.pop()
        try:
            with os.scandir(current) as entries:
                for entry in entries:
                    try:
                        if entry.is_dir(follow_symlinks=False):
                            stack.append(Path(entry.path))
                        elif entry.is_file(follow_symlinks=False):
                            yield Path(entry.path), entry.stat(follow_symlinks=False).st_size
                    except OSError as exc:
                        print(f"Skipping unreadable entry {entry.path}: {exc}")
        except OSError as exc:
            print(f"Skipping unreadable directory {current}: {exc}")

def directory_size(root):
    total = 0
    file_count = 0
    for _, size in iter_files(root):
        total += size
        file_count += 1
    return total, file_count

In [ ]:
if not CELL_DATA_ROOT.exists():
    raise FileNotFoundError(f"External dataset root not found: {CELL_DATA_ROOT}")

date_dirs = sorted([path for path in CELL_DATA_ROOT.iterdir() if path.is_dir()], key=lambda p: p.name)
print(f"Date folders found: {len(date_dirs):,}")

rows = []
for date_dir in date_dirs:
    size_bytes, file_count = directory_size(date_dir)
    rows.append(
        {
            "date": date_dir.name,
            "file_count": file_count,
            "size_bytes": size_bytes,
            "size_gb": size_bytes / 1_000_000_000,
            "size_tib": size_bytes / 1024**4,
        }
    )
    print(f"{date_dir.name}: {file_count:,} files, {format_size(size_bytes)}")

date_size_df = pd.DataFrame(rows)
total_size = int(date_size_df["size_bytes"].sum()) if not date_size_df.empty else 0
total_files = int(date_size_df["file_count"].sum()) if not date_size_df.empty else 0

print("\nTotal external dataset size")
print(f"Files: {total_files:,}")
print(f"Size:  {format_size(total_size)}")

date_size_df

## Annotated Data Size

The annotated size is calculated from `Descriptor_fs.csv`. For every annotated `(date, cell_id)` pair, the notebook checks the five expected files: `r`, `g`, `b`, `s`, and `d`. Missing files are reported but not copied.

In [ ]:
if not DESCRIPTOR_PATH.exists():
    raise FileNotFoundError(f"Descriptor file not found: {DESCRIPTOR_PATH}")

# Descriptor_fs.csv in this project is semicolon-separated and has no header:
# date;cell_id;class
try:
    descriptor = pd.read_csv(DESCRIPTOR_PATH, dtype=str, sep=None, engine="python", header=None, encoding="utf-8-sig")
except UnicodeDecodeError:
    descriptor = pd.read_csv(DESCRIPTOR_PATH, dtype=str, sep=None, engine="python", header=None)

if descriptor.shape[1] < 2:
    raise ValueError(
        f"Descriptor_fs.csv must contain at least date and cell_id columns; got shape {descriptor.shape}."
    )

descriptor = descriptor.iloc[:, :3].copy()
descriptor.columns = ["date", "cell_id", "class"][: descriptor.shape[1]]

annotated = descriptor[["date", "cell_id"]].dropna().copy()
annotated["date"] = annotated["date"].astype(str).str.replace("\ufeff", "", regex=False).str.strip()
annotated["cell_id"] = annotated["cell_id"].astype(str).str.strip()
annotated = annotated.drop_duplicates()

print(f"Annotated cells in descriptor: {len(annotated):,}")
print(f"Dates with annotated cells:    {annotated['date'].nunique():,}")
display(annotated.head())

In [ ]:
annotated_rows = []
missing_rows = []

for row in annotated.itertuples(index=False):
    cell_total = 0
    files_found = 0
    for suffix in FEATURE_SUFFIXES:
        file_path = CELL_DATA_ROOT / row.date / f"{row.cell_id}{suffix}.txt"
        try:
            stat = file_path.stat()
        except FileNotFoundError:
            missing_rows.append({"date": row.date, "cell_id": row.cell_id, "feature": suffix, "path": str(file_path)})
            continue
        except OSError as exc:
            missing_rows.append({"date": row.date, "cell_id": row.cell_id, "feature": suffix, "path": str(file_path), "error": str(exc)})
            continue
        cell_total += stat.st_size
        files_found += 1

    annotated_rows.append(
        {
            "date": row.date,
            "cell_id": row.cell_id,
            "files_found": files_found,
            "size_bytes": cell_total,
        }
    )

annotated_size_df = pd.DataFrame(annotated_rows)
missing_annotated_files_df = pd.DataFrame(missing_rows)

annotated_by_date_df = (
    annotated_size_df
    .groupby("date", as_index=False)
    .agg(
        annotated_cells=("cell_id", "count"),
        files_found=("files_found", "sum"),
        size_bytes=("size_bytes", "sum"),
    )
)
annotated_by_date_df["size_gb"] = annotated_by_date_df["size_bytes"] / 1_000_000_000
annotated_by_date_df["size_tib"] = annotated_by_date_df["size_bytes"] / 1024**4

annotated_total_size = int(annotated_by_date_df["size_bytes"].sum()) if not annotated_by_date_df.empty else 0
annotated_total_files = int(annotated_by_date_df["files_found"].sum()) if not annotated_by_date_df.empty else 0

print("Annotated external data size")
print(f"Files found: {annotated_total_files:,}")
print(f"Size:        {format_size(annotated_total_size)}")
print(f"Missing expected annotated files: {len(missing_annotated_files_df):,}")

display(annotated_by_date_df)
if not missing_annotated_files_df.empty:
    display(missing_annotated_files_df.head(50))